# Rwanda WorldCover LULC embedding check

A lightweight quality check for the WorldCover non-crop TESSERA run. It is safe to run while inference is active: it considers only atomically published Parquet shards, validates their metadata and schema, and never treats a missing `COMPLETED.json` as an error.

The plots diagnose the learned representation and source coverage. WorldCover labels are weak 2021 map labels, not independent ground truth.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_DIR = Path(os.environ.get(
    'TESSERA_OUTPUT_DIR', '/mnt/noobjam/rwanda_worldcover_mlp/tessera_embeddings'
))
MAX_PCA_SAMPLES_PER_CLASS = 500
RANDOM_SEED = 20260710

print('Output:', OUTPUT_DIR)
print('Override with TESSERA_OUTPUT_DIR if needed.')

In [ ]:
run_path = OUTPUT_DIR / 'run.json'
if not run_path.is_file():
    raise FileNotFoundError(f'No run manifest at {run_path}')
run = json.loads(run_path.read_text())
fingerprint = run.get('run_fingerprint')
if not fingerprint:
    raise RuntimeError('run.json has no run_fingerprint')

required_columns = {
    'row_id', 'run_fingerprint', 'field_uid', 'source_id', 'landcover', 'pixel_id',
    'pixel_longitude', 'pixel_latitude', 'window_id', 'outcome', 'embedding',
    's1_valid_count', 's2_valid_count', 's1_input_count', 's2_input_count',
}
files = sorted(path for path in (OUTPUT_DIR / 'embeddings').glob('window_id=w*/*.parquet')
               if not path.name.endswith('.part'))
if not files:
    raise RuntimeError('No atomically published embedding shards are available yet.')

parts = []
for path in files:
    parquet = pq.ParquetFile(path)
    metadata = {key.decode(): value.decode() for key, value in (parquet.metadata.metadata or {}).items()}
    window_id = path.parent.name.split('=', 1)[1]
    expected = {'run_fingerprint': fingerprint, 'artifact': 'pixel_embeddings',
                'schema_version': '1', 'window_id': window_id}
    mismatched = {key: (metadata.get(key), value) for key, value in expected.items()
                  if metadata.get(key) != value}
    if mismatched:
        raise RuntimeError(f'Metadata mismatch in {path}: {mismatched}')
    missing = required_columns - set(parquet.schema_arrow.names)
    if missing:
        raise RuntimeError(f'Missing columns in {path}: {sorted(missing)}')
    embedding_type = parquet.schema_arrow.field('embedding').type
    if not pa.types.is_list(embedding_type) or embedding_type.value_type != pa.float32():
        raise RuntimeError(f'Unexpected embedding type in {path}: {embedding_type}')
    part = parquet.read(columns=sorted(required_columns)).to_pandas()
    if not part['window_id'].eq(window_id).all():
        raise RuntimeError(f'Rows do not match the window partition in {path}')
    parts.append(part)

rows = pd.concat(parts, ignore_index=True)
if rows['row_id'].duplicated().any():
    raise RuntimeError('Duplicate row_id values found across published shards.')
if not rows['run_fingerprint'].eq(fingerprint).all():
    raise RuntimeError('Published rows belong to more than one run.')
rows['embedding_ok'] = rows['embedding'].map(
    lambda value: value is not None and (vector := np.asarray(value, dtype=np.float32)).shape == (128,)
    and np.isfinite(vector).all()
)
if (rows['outcome'].eq('complete') & ~rows['embedding_ok']).any():
    raise RuntimeError('A complete row has no finite float32[128] embedding.')
if (rows['outcome'].ne('complete') & rows['embedding'].notna()).any():
    raise RuntimeError('A non-complete row unexpectedly has an embedding.')

windows = sorted(rows['window_id'].unique(), key=lambda value: int(value.removeprefix('w')))
print('Run complete:', (OUTPUT_DIR / 'COMPLETED.json').is_file())
print('Published shards:', len(files), '| rows:', len(rows), '| windows:', windows)
print('Expected final rows:', run.get('expected_embedding_rows', 'unknown'))

In [ ]:
progress = (rows.assign(complete=rows['outcome'].eq('complete'))
    .groupby(['window_id', 'landcover'], sort=True)
    .agg(rows=('row_id', 'size'), unique_pixels=('pixel_id', 'nunique'),
         complete=('complete', 'sum'), valid_embeddings=('embedding_ok', 'sum'),
         median_s2_valid=('s2_valid_count', 'median'), median_s1_valid=('s1_valid_count', 'median'))
    .reset_index())
progress['complete_rate'] = progress['complete'] / progress['rows']
progress['valid_embedding_rate'] = progress['valid_embeddings'] / progress['rows']
display(progress.sort_values(['window_id', 'landcover']))

expected = run.get('expected_embedding_rows')
if expected:
    print(f'Published progress: {len(rows):,} / {expected:,} rows ({len(rows) / expected:.1%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
pivot = progress.pivot(index='landcover', columns='window_id', values='rows').fillna(0)
pivot.plot(kind='bar', ax=axes[0])
axes[0].set_title('Published rows by WorldCover class')
axes[0].set_ylabel('rows'); axes[0].set_xlabel('')
rate = progress.pivot(index='landcover', columns='window_id', values='complete_rate').fillna(0)
rate.plot(kind='bar', ax=axes[1], ylim=(0, 1))
axes[1].set_title('Complete-embedding rate')
axes[1].set_ylabel('rate'); axes[1].set_xlabel('')
plt.show()

In [ ]:
TARGET_WINDOW = 'w2' if 'w2' in windows else windows[-1]
complete = rows[rows['window_id'].eq(TARGET_WINDOW) & rows['embedding_ok']].copy()
if len(complete) < 3:
    raise RuntimeError(f'Need at least three finite embeddings in {TARGET_WINDOW}; found {len(complete)}.')
sampled = (complete.groupby('landcover', group_keys=False, sort=True)
    .apply(lambda group: group.sample(min(len(group), MAX_PCA_SAMPLES_PER_CLASS), random_state=RANDOM_SEED))
    .reset_index(drop=True))
vectors = np.vstack(sampled['embedding'].map(lambda value: np.asarray(value, dtype=np.float32)))
centered = vectors - vectors.mean(axis=0, keepdims=True)
_, singular_values, right_vectors = np.linalg.svd(centered, full_matrices=False)
if (singular_values > 1e-8).sum() < 2:
    raise RuntimeError('Sampled embeddings do not support a two-dimensional PCA diagnostic.')
coordinates = centered @ right_vectors[:2].T
sampled['pc1'], sampled['pc2'] = coordinates[:, 0], coordinates[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for label, group in sampled.groupby('landcover', sort=True):
    axes[0].scatter(group['pc1'], group['pc2'], s=10, alpha=0.65, label=label)
    axes[1].scatter(group['pixel_longitude'], group['pixel_latitude'], s=8, alpha=0.6, label=label)
axes[0].set(title=f'{TARGET_WINDOW}: sampled 128-D embeddings (PCA)', xlabel='PC1', ylabel='PC2')
axes[1].set(title=f'{TARGET_WINDOW}: locations of the same sampled pixels', xlabel='longitude', ylabel='latitude')
for axis in axes:
    axis.grid(alpha=0.2)
    axis.legend(title='WorldCover class', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

print(f'PCA sample: {len(sampled):,} finite embeddings across {sampled.landcover.nunique()} classes.')

## Reading the check

- A missing `COMPLETED.json` simply means this is a live snapshot; rerun later to refresh it.
- `complete_rate` below 1 is expected where a temporal prefix has no usable S1/S2 observations; those rows are preserved as `empty_window` and have no vector.
- PCA proximity is only a qualitative embedding diagnostic. It does not establish LULC accuracy or validate WorldCover labels.
- Once the run completes, use `build_pixel_classification_dataset` to merge these non-crop embeddings with the Harvard pure-crop run and perform the spatial split.